# The *sommerhus*, characterized dynamically

Notebook 3 gave the house a **timeline**. This notebook asks what the timing of its emissions
does to the **impact**, with the dynamic characterization from the previous session - now on a
real inventory instead of a handful of dummy rows.

Two things are yours to work out: how the **time horizon** is counted (section 3), and a
**characterization function of your own** (section 4).

```mermaid
flowchart LR
    tree(🌲 timber tree):::fg-->timber
    timber(🪵 structural timber production):::fg-->construction
    glass_wool(🧵 market for glass wool mat):::ei-->construction
    construction(🏗️ sommerhus construction):::fg-->living
    heat_pump(🔥 heat pump, brine-water 10kW):::ei-->living
    grid(⚡ market group for electricity, low voltage):::ei-->living
    living(🏠 living in the sommerhus):::fg-->waste_wood(🔥 incineration of waste wood):::fg
    living-->fu(FU: 50 years of habitation)

    classDef ei color:#222832, fill:#3fb1c5, stroke:none;
    classDef fg color:#222832, fill:#9c5ffd, stroke:none;
```

<span style="color:#9c5ffd">■</span> foreground &nbsp;&nbsp;
<span style="color:#3fb1c5">■</span> background (premise vintages 2020 / 2030 / 2040 / 2050)



## 0 | The model from notebook 3

System and temporal information are imported rather than retyped -
[`sommerhus_system.py`](sommerhus_system.py) and
[`sommerhus_temporal.py`](sommerhus_temporal.py) hold exactly what you wrote by hand before.


In [ ]:
import bw2data as bd

from sommerhus_system import build_system
from sommerhus_temporal import add_temporal_information

LIFETIME = 50
BG_DATABASE = "ei_cutoff_3.12_remind-eu_SSP2-NDC_2020"
METHOD = ("IPCC 2021", "climate change", "GWP 100a, incl. H and bio CO2")

build_system(
    lifetime=LIFETIME,
    timber_volume=12,
    insulation_mass=1200,
    heat_pumps=3,
    electricity_per_year=1400,
    background_database=BG_DATABASE,
    method=METHOD,
)
add_temporal_information(lifetime=LIFETIME, background_database=BG_DATABASE)

living = bd.get_node(database="foreground", name="living in the sommerhus")

In [ ]:
from bw_timex import TimexLCA

tlca = TimexLCA({living: 1}, METHOD)
tlca.build_timeline(starting_datetime="2025-01-01", temporal_grouping="month")
tlca.lci()
tlca.static_lcia()

print(f"time-explicit, static characterization: {tlca.static_score:,.0f} kg CO2-eq")

## 1 | The inventory, before any characterization

`tlca.dynamic_inventory_df` is the `date` / `amount` / `flow` / `activity` dataframe you
already know - only this one comes out of a real supply chain. The tree's uptake sits decades
before the house exists. (Positive here means *taken out of the air*: this is a natural
resource flow, and the sign flips once it is characterized.)


In [ ]:
import matplotlib.pyplot as plt

uptake_flow = bd.get_node(
    database=bd.config.biosphere,
    name="Carbon dioxide, in air",
    categories=("natural resource", "in air"),
)
uptake_over_time = (
    tlca.dynamic_inventory_df[tlca.dynamic_inventory_df["flow"] == uptake_flow.id]
    .groupby("date")["amount"]
    .sum()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(uptake_over_time.index, uptake_over_time.values, marker="o", linestyle="none")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("kg CO2 taken up")
plt.tight_layout()
plt.show()

## 2 | Radiative forcing over time

`metric="radiative_forcing"` characterizes each emission from the moment it happens, without
integrating anything away. `bw_timex` maps the IPCC AR6 functions to the biosphere flows of
`METHOD` by itself, so there is no `characterization_functions` dict to pass here - we are on
ecoinvent flows.


In [ ]:
# instantaneous, one series per emitting activity
tlca.dynamic_lcia(metric="radiative_forcing", time_horizon=100)
tlca.plot_dynamic_characterized_inventory(sum_emissions_within_activity=True)

The same characterization, now summed over all activities and **accumulated** - the
warming the house has caused up to each point in time, rather than in each single year:


In [ ]:
tlca.plot_dynamic_characterized_inventory(sum_activities=True, cumsum=True)

The house life cycle spends its first decades **cooling**: the tree's uptake is characterized
before anything is emitted. The curve crosses zero only years after the house's construction.


## 3 | 🛠️ Your turn: fixed or flexible time horizon?

Both options say "100 years", and they mean different things:

| | window for one emission |
|---|---|
| `fixed_time_horizon=False` (default) | 100 years starting **at that emission** |
| `fixed_time_horizon=True` (Levasseur) | up to the **functional unit's date + 100 years**, whoever emits |

This system is the interesting case, because its flows are spread over more than a century:
the tree takes up CO2 up to 40 years *before* the functional unit, the incineration happens
50 years *after* it.

**Predict first, then run:** does `fixed_time_horizon=True` give a higher or a lower GWP100
than the default - and why?


In [ ]:
# TODO: compute the GWP100 of this system both ways and print the two scores.
#       One call each: tlca.dynamic_lcia(metric=..., time_horizon=..., fixed_time_horizon=...),
#       and read the result off tlca.dynamic_score.

In [ ]:
# Stuck, or out of time? Uncomment the line below, run this cell twice,
# and you're back in sync with everyone else.
# %load solutions/5_sommerhus_dynamic/1_fixed_time_horizon.py

The same two variants as **radiative forcing over time**, in two panels because the
two quantities differ by a factor of ~70 and would hide each other on one axis:

- **top**: the radiative forcing *in* each year - the instantaneous warming effect of
  everything emitted so far, as it decays.
- **bottom**: the running sum of the top panel. This is the area a GWP integrates into its
  single number, which is why the two curves end where the two GWP100 scores were.

The dotted lines are move-in (2025) and the common cut-off the fixed horizon uses
(2125 = functional unit + 100 years):


In [ ]:
import pandas as pd

series = {}
for fixed in (False, True):
    tlca.dynamic_lcia(metric="radiative_forcing", time_horizon=100, fixed_time_horizon=fixed)
    rf = tlca.characterized_inventory.groupby("date")["amount"].sum().sort_index()
    series[fixed] = rf.resample("YS").sum()  # monthly resolution is noise at this scale

# flexible drawn solid and underneath, fixed dashed on top: wherever the two agree, the
# dashes sit straight on the blue line
STYLES = {
    False: dict(color="tab:blue", linewidth=2.2, linestyle="-",
                label="flexible: 100 years from each emission"),
    True: dict(color="tab:orange", linewidth=1.6, linestyle="--",
               label="fixed: everything counted until 2125"),
}

fig, (ax_year, ax_cum) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

for fixed, style in STYLES.items():
    rf = series[fixed]
    ax_year.plot(rf.index, rf.values, **style)
    ax_cum.plot(rf.index, rf.cumsum().values, **style)

ax_year.set_title("instantaneous - radiative forcing in each year")
ax_cum.set_title("cumulative - the running sum, i.e. what a GWP integrates")

for ax in (ax_year, ax_cum):
    ax.axhline(0, color="black", linewidth=0.8)
    bottom = ax.get_ylim()[0]
    for year, label in [(2025, "move-in"), (2125, "FU + 100 a")]:
        date = pd.Timestamp(f"{year}-01-01")
        ax.axvline(date, color="grey", linestyle=":", linewidth=1)
        ax.text(date, bottom, f" {label}", color="grey", fontsize=9, va="bottom")
    ax.set_ylabel("W/m2")
    ax.legend(loc="upper left", framealpha=0.9)

plt.tight_layout()
plt.show()

For most of the century the dashes sit straight on the solid line: every flow is
still inside both windows, so both runs characterize it identically. They come apart around
2100 in two steps.

First the dashed line drops **below** the solid one. That is the tree: its uptake is negative
forcing, the flexible run closes the 100-year window on it around 2085, and the fixed run
keeps counting it to 2125 - about 140 years of credit instead of 100. Then, at 2125, the
fixed run stops altogether, while the flexible one carries the 2075 incineration on to 2174.

Cooling counted longer, warming counted shorter: that is the whole of the 24,711 vs 13,704
gap. This value corresponds to the integral of the curve in the bottom panel.



### 🛠️ Your turn: how much does the horizon length itself matter?

Use [`TimexLCA.compare()`](https://docs.brightway.dev/projects/bw-timex/en/latest/content/api/bw_timex/timex_lca/index.html)
to characterize the sommerhus inventory with GWP over **20, 50, 100 and 500 years, each with `fixed_time_horizon` False and True**.

**Predict first, then run:** which of the eight numbers comes out **negative**, and what
happens to the gap between the two columns as the horizon grows?


In [ ]:
# TODO: run TimexLCA.compare() with different TimexLCASettings:
#       time_horizon 20 / 50 / 100 / 500, each with fixed_time_horizon False and True.
#       Start from one base TimexLCASettings (demand={living: 1}, method=METHOD,
#       starting_datetime="2025-01-01", temporal_grouping="month", metric="GWP") and
#       dataclasses.replace() it per run. See also the last part of the 3_sommerhus.ipynb notebook.

In [ ]:
# Stuck, or out of time? Uncomment the line below, run this cell twice,
# and you're back in sync with everyone else.
# %load solutions/5_sommerhus_dynamic/2_time_horizon_sweep.py

## 4 | 🛠️ Your turn: a characterization function of your own

Nothing in `characterize()` is climate-specific - it applies whatever function you map onto a
flow. So let us give the *sommerhus* something that climate metrics cannot see.

A Danish summer house draws its water from **its own well**, and it is lived in during the
**summer** - exactly when the groundwater is under most pressure. Static LCIA has one factor per flow and cannot express that, but a dynamic characterization
function can.

**(a) Add a water flow to the model.** 60 m3 of `"Water, well, in ground"` per year of
habitation, drawn **April to September** - the house fills up and the garden needs watering -
with the peak in **July and August**. Say:

| Apr | May | Jun | Jul | Aug | Sep |
|---|---|---|---|---|---|
| 7% | 12% | 20% | 27% | 24% | 10% |

Then rebuild the `TimexLCA` so the inventory contains it.

> The flow lives in `bd.config.biosphere`, categories `("natural resource", "in water")`.
> A biosphere edge is `living.new_edge(input=..., amount=..., type="biosphere")`, and its
> `temporal_distribution` works exactly like the ones on technosphere edges: months relative
> to the consumer, shares that sum to 1 **over the whole exchange**. 


In [ ]:
# TODO: add the water flow to `living` (see the hints above), give it a temporal
#       distribution that spreads each year's withdrawal over the season, and rebuild
#       the TimexLCA:
#       TimexLCA(...) -> build_timeline(starting_datetime="2025-01-01",
#       temporal_grouping="month") -> lci()

In [ ]:
# Stuck, or out of time? Uncomment the line below, run this cell twice,
# and you're back in sync with everyone else.
# %load solutions/5_sommerhus_dynamic/3_water_flow.py

**(b) Characterize it.** You already wrote this function: `characterize_water_scarcity`
from [`4_dynamic_characterization.ipynb`](4_dynamic_characterization.ipynb), with
`water_stress_index_by_month`. Copy both across - nothing about them changes here. The only new part is what you point it at.

> Call `characterize()` on `tlca.dynamic_inventory_df` directly, with
> `characterization_functions={water_flow.id: characterize_water_scarcity}`.


In [ ]:
# TODO: bring `characterize_water_scarcity` and `water_stress_index_by_month` over from the
#       4_dynamic_characterization.ipynb - they work here unchanged - and apply them to
#       tlca.dynamic_inventory_df.

In [ ]:
# Stuck, or out of time? Uncomment the line below, run this cell twice,
# and you're back in sync with everyone else.
# %load solutions/5_sommerhus_dynamic/4_water_characterization.py

## 5 | 🚀 Outlook, for the ambitious: make it prospective

The index you just used is a snapshot of *today's* summer. It will not hold for 2075: drier
summers, a lower water table, more neighbours on the same aquifer. The climate functions have
the same problem, and the Watanabe pCFs from the previous notebook solve it by letting the
characterization factor depend on the **year** of the emission as well as the substance.

Do the same for the water. Write `characterize_water_scarcity_prospective(series, period=1)`
that keeps the month lookup and multiplies it by a trend read off the year - say stress rises
by 60% between 2025 and 2075 and is held flat outside that range - then characterize the same
inventory with it and compare.

> `np.interp(year, [2025, 2075], [1.0, 1.6])` gives you the trend and clamps outside the
> range, exactly like the temporal evolution factors in notebook 3. The month must survive:
> `series.date` still carries it, so keep reading `series.date.month` and do **not** route
> this through `dynamic_lcia()`.
>
> Worth asking yourself afterwards: this house draws the same 60 m3 every summer for 50 years.
> Under a rising index, is its *late* water worth more than its early water - and what would
> that mean for a house built in 2045 instead?


In [ ]:
# TODO: write characterize_water_scarcity_prospective(series, period=1) - the same month
#       lookup, times np.interp(series.date.year, [2025, 2075], [1.0, 1.6]) - and
#       characterize tlca.dynamic_inventory_df with it.

In [ ]:
# Stuck, or out of time? Uncomment the line below, run this cell twice,
# and you're back in sync with everyone else.
# %load solutions/5_sommerhus_dynamic/5_water_prospective.py

In [ ]:
if "water_scarcity_prospective" in globals():
    per_year = {
        "today's index": water_scarcity.assign(year=water_scarcity["date"].dt.year)
        .groupby("year")["amount"].sum(),
        "rising index": water_scarcity_prospective.assign(
            year=water_scarcity_prospective["date"].dt.year
        ).groupby("year")["amount"].sum(),
    }

    fig, ax = plt.subplots(figsize=(11, 3.5))
    for label, series_ in per_year.items():
        ax.plot(series_.index, series_.values, marker="o", markersize=3, label=label)
    ax.set_xlabel("year of withdrawal")
    ax.set_ylabel("stress-weighted m3 per year")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("nothing to compare yet - section 5 is still open")

## 6 | Where to go from here

- **Prospective characterization factors** (Watanabe et al. 2026): same call, `metric="pGWP"`
  or `"prospective_radiative_forcing"` after `prospective.set_scenario(...)`. On a system whose
  emissions run to 2125, the scenario-dependent radiative efficiencies are not a detail.
- **`fixed_time_horizon` in a comparison**: two houses built decades apart are exactly the case
  where the choice changes the ranking, not just the number.
- **Your own metric**: the water function above is 8 lines. Anything that depends on *when* -
  seasonal water, noise at night, harvest timing - fits the same shape.
